# Lab 2: Vectorless RAG — Advanced Scenarios

## Setup

### Install Dependencies

In [5]:
# Install PageIndex for vectorless hierarchical retrieval, LangChain OpenRouter for the LLM, and requests for downloading the file.
!pip install pageindex langchain-openrouter requests


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import Libraries

In [26]:
# Import core modules
import os
import time
import requests

# Import PageIndex for vectorless retrieval
from pageindex import PageIndexClient
import pageindex.utils as utils
from dotenv import load_dotenv

# Import LangChain's OpenRouter wrapper
from langchain_groq import ChatGroq

### Setup API Keys

In [28]:
load_dotenv("../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")

# Initialize the PageIndex Client
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

---
## Load and Parse the PDF

### Define PDF Path

In [13]:
import os

# Point to the local PDF document you already have
PDF_PATH = "data/CCS 3.31.25 Earnings Release 8-K Exhibit 99.1.pdf"

# Let's add a quick beginner-friendly check to make sure the file exists where we expect it
if os.path.exists(PDF_PATH):
    print(f"Success: Found the document at '{PDF_PATH}'")
else:
    print(f"Error: Could not find the document. Please make sure the 'data' folder is in the same directory as this notebook.")

Success: Found the document at 'data/CCS 3.31.25 Earnings Release 8-K Exhibit 99.1.pdf'


### Submit and Index Document (Tree Construction)

In [14]:
# Submit your local document to PageIndex. 
# It reads your financial document and organizes it into a logical tree (Sections, Tables, Text).
doc_info = pi_client.submit_document(PDF_PATH)
doc_id = doc_info["doc_id"]

print(f"Document Submitted. Tracking ID: {doc_id}")

# Polling loop: Wait for the document tree to finish building
print("Waiting for the document to be indexed...")
while not pi_client.is_retrieval_ready(doc_id):
    print("Still processing... checking again in 5 seconds.")
    time.sleep(5)

print("Indexing Complete! The hierarchical tree is ready for multi-hop retrieval.")

Document Submitted. Tracking ID: pi-cmrmwiexy00l101o46ca8q8c0
Waiting for the document to be indexed...
Still processing... checking again in 5 seconds.
Still processing... checking again in 5 seconds.
Still processing... checking again in 5 seconds.
Indexing Complete! The hierarchical tree is ready for multi-hop retrieval.


### Initialize OpenRouter LLM

In [57]:
# Initialize the Large Language Model via OpenRouter
# OpenRouter gives you access to multiple models. Here we use an open model as an example.
llm = ChatGroq(
    model="llama-3.1-8b-instant", 
    temperature=0.0,
    max_tokens=300  
)

### Define Retrieval Function (With Explainability Tracking)

In [58]:
def retrieve_from_pageindex(query, doc_id, top_k=3):
    # 1. Submit the user's question
    response = pi_client.submit_query(doc_id=doc_id, query=query)
    retrieval_id = response.get("retrieval_id")
    
    if not retrieval_id:
        return [], []
        
    # 2. Polling Loop: Wait until the search finishes
    while True:
        retrieval = pi_client.get_retrieval(retrieval_id)
        status = retrieval.get("status")
        
        if status == "completed":
            break
        elif status == "failed":
            return [], []
            
        time.sleep(1)
        
    # 3. Extract the text AND build the tracking log
    nodes = retrieval.get("retrieved_nodes", [])
    contexts = []
    trace_path = [] # <-- We will store our reasoning hops here
    
    # Loop through the top results
    for index, node in enumerate(nodes[:top_k]):
        
        # --- SIMPLE FIX 1: Get the real title, or default to "Section 1"
        node_name = node.get("node_title") or node.get("title") or f"Section {index + 1}"
        
        # Extract the actual readable text
        relevant_contents = node.get("relevant_contents", [])
        for group in relevant_contents:
            for item in group:
                content = item.get("relevant_content")
                if content:
                    contexts.append(content)
                    
        # Grab a quick 50-character preview of the text we just extracted
        preview = contexts[-1][:50].replace('\n', ' ') if contexts else "No text found"
        
        # Log this hop for explainability
        trace_path.append(f"Hop {index + 1}: {node_name} -> '{preview}...'")
                    
    # Return BOTH the text (for the LLM) and the trace (for the human)
    return contexts, trace_path

In [67]:
def vectorless_rag(query, doc_id):
    # Get both the context text AND our tracking log
    contexts, trace_path = retrieve_from_pageindex(query, doc_id)
    
    if not contexts:
        return "No relevant context found.", []
        
    # Combine the text for the LLM
    combined_context = "\n\n".join(contexts)
    
    # Give the OpenRouter LLM strict instructions
    prompt = f"""
You are a financial analyst. Answer the question using ONLY the data in the context below.

CRITICAL INSTRUCTIONS:
- If the question requires a calculation or comparison (like checking if a pace supports a goal), you ARE ALLOWED to perform basic math using the numbers found in the context.
- Show your math briefly, then state your final conclusion.
- If the required numbers are missing, say "Not found in document."

Context: {combined_context}

Question: {query}
"""
    
    # Generate the answer using OpenRouter
    response = llm.invoke(prompt)
    
    # Return the AI's answer AND the explainability trace
    return response.content, trace_path

### Run Query and Show the Tracking

In [68]:
# The question we want to ask
query = "Does the pace of home deliveries in Q1 2025 support the company's full-year 2025 guidance?"

In [69]:
print(f"Question: {query}\n")
print("Navigating document tree and generating answer...\n")

# Run our updated function
final_answer, trace_path = vectorless_rag(query, doc_id)

# --- EXPLAINABILITY TRACKING ---
print("--- SYSTEM REASONING TRACE (HOW IT FOUND THE ANSWER) ---")
if trace_path:
    for step in trace_path:
        print(step)
else:
    print("No hops recorded.")

Question: Does the pace of home deliveries in Q1 2025 support the company's full-year 2025 guidance?

Navigating document tree and generating answer...

--- SYSTEM REASONING TRACE (HOW IT FOUND THE ANSWER) ---
Hop 1: Full Year 2025 Outlook -> 'Scott Dixon, Chief Financial Officer of the Compan...'
Hop 2: **Home Deliveries**(dollars in thousands) -> 'Home Deliveries (dollars in thousands) for the thr...'


In [70]:
# --- FINAL AI ANSWER ---
print("\n--- FINAL LLM ANSWER ---")
print(final_answer)


--- FINAL LLM ANSWER ---
To determine if the pace of home deliveries in Q1 2025 supports the company's full-year 2025 guidance, we need to calculate the quarterly pace and compare it to the full-year guidance.

The full-year guidance is 10,400 to 11,000 homes.

The total home deliveries for Q1 2025 are 2,284 homes.

To calculate the quarterly pace, we divide the Q1 deliveries by 4:
2,284 homes / 4 = 571 homes per quarter.

Since the full-year guidance is 10,400 to 11,000 homes, we can calculate the required quarterly pace:
10,400 homes / 4 = 2,600 homes per quarter (lower end of guidance)
11,000 homes / 4 = 2,750 homes per quarter (upper end of guidance)

Comparing the calculated quarterly pace of 571 homes to the required pace of 2,600 to 2,750 homes per quarter, we can conclude that the pace of home deliveries in Q1 2025 does not support the company's full-year 2025 guidance.
